# 🎬 MovieLens Data Cleaning
**Role:** Data Engineer / Data Prep Lead  
**Dataset:** MovieLens Latest Small (100K ratings)  
**Output:** Clean CSVs in `/data/processed/`

---
This notebook handles:
- Downloading and loading the raw dataset
- Auditing for missing values, duplicates, and bad IDs
- Standardizing formats
- Engineering base features (avg rating, count, popularity score, genre encoding)
- Train / validation / test split
- Saving clean artifacts for the modeling team


## 1. Imports & Setup

In [1]:
import os, zipfile, urllib.request
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from scipy.sparse import csr_matrix

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid", palette="muted")
print("Libraries loaded ✓")


Libraries loaded ✓


## 2. Download MovieLens Dataset

In [2]:
RAW_DIR = "../data/raw"
ZIP_PATH = os.path.join(RAW_DIR, "ml-latest-small.zip")
EXTRACT_DIR = os.path.join(RAW_DIR, "ml-latest-small")
URL = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"

os.makedirs(RAW_DIR, exist_ok=True)

if not os.path.exists(ZIP_PATH):
    print("Downloading ml-latest-small.zip ...")
    urllib.request.urlretrieve(URL, ZIP_PATH)
    print("Download complete ✓")
else:
    print("Zip already present, skipping download.")

if not os.path.exists(EXTRACT_DIR):
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(RAW_DIR)
    print("Extracted ✓")
else:
    print("Files already extracted.")

print("\nFiles in dataset:")
for f in os.listdir(EXTRACT_DIR):
    print(f"  {f}")


PermissionError: [Errno 13] Permission denied: '../data'

## 3. Load Raw Files

In [ ]:
DATA_DIR = EXTRACT_DIR

ratings  = pd.read_csv(os.path.join(DATA_DIR, "ratings.csv"))
movies   = pd.read_csv(os.path.join(DATA_DIR, "movies.csv"))
tags     = pd.read_csv(os.path.join(DATA_DIR, "tags.csv"))
links    = pd.read_csv(os.path.join(DATA_DIR, "links.csv"))

print("Shape of each file:")
for name, df in [("ratings", ratings), ("movies", movies), ("tags", tags), ("links", links)]:
    print(f"  {name:10s}: {df.shape[0]:,} rows × {df.shape[1]} cols")


In [ ]:
# Quick peek
print("--- ratings ---"); display(ratings.head(3))
print("--- movies ---");  display(movies.head(3))
print("--- tags ---");    display(tags.head(3))


## 4. Data Quality Audit

In [ ]:
def audit(df, name):
    print(f"\n{'='*40}")
    print(f"  {name}")
    print(f"{'='*40}")
    print(f"Shape        : {df.shape}")
    print(f"Duplicates   : {df.duplicated().sum()}")
    print("\nNull counts:")
    print(df.isnull().sum().to_string())
    print("\nDtypes:")
    print(df.dtypes.to_string())

audit(ratings, "ratings")
audit(movies,  "movies")
audit(tags,    "tags")


In [ ]:
# Rating range check
print("Rating value counts:")
print(ratings["rating"].value_counts().sort_index())
print(f"\nMin: {ratings['rating'].min()}  Max: {ratings['rating'].max()}")
assert ratings["rating"].between(0.5, 5.0).all(), "Out-of-range ratings found!"
print("\nAll ratings in valid range [0.5 – 5.0] ✓")


In [ ]:
# Referential integrity: every movieId in ratings must exist in movies
rating_movie_ids = set(ratings["movieId"].unique())
movie_ids        = set(movies["movieId"].unique())
orphans = rating_movie_ids - movie_ids
print(f"movieIds in ratings but NOT in movies: {len(orphans)}")
if orphans:
    print("  Sample orphan IDs:", list(orphans)[:10])


## 5. Cleaning

In [ ]:
# --- ratings ---
ratings_clean = ratings.copy()
ratings_clean.drop_duplicates(inplace=True)

# Convert timestamp to datetime
ratings_clean["timestamp"] = pd.to_datetime(ratings_clean["timestamp"], unit="s")
ratings_clean.rename(columns={"timestamp": "rated_at"}, inplace=True)

# Drop orphan ratings (movies with no metadata)
ratings_clean = ratings_clean[ratings_clean["movieId"].isin(movie_ids)]

print(f"ratings rows: {len(ratings):,}  →  {len(ratings_clean):,} after cleaning")


In [ ]:
# --- movies ---
movies_clean = movies.copy()
movies_clean.drop_duplicates(inplace=True)

# Parse year from title  (e.g. "Toy Story (1995)")
movies_clean["year"] = (
    movies_clean["title"]
    .str.extract(r"\((\d{4})\)")
    .astype("Int64")     # nullable int — keeps NaN for missing years
)
movies_clean["title_clean"] = movies_clean["title"].str.replace(r"\s*\(\d{4}\)", "", regex=True).str.strip()

# Split genres into a list column (keep original string too)
movies_clean["genre_list"] = movies_clean["genres"].apply(
    lambda g: g.split("|") if g != "(no genres listed)" else []
)

print(f"movies rows: {len(movies):,}  →  {len(movies_clean):,} after cleaning")
display(movies_clean.head(3))


In [ ]:
# --- tags ---
tags_clean = tags.copy()
tags_clean.drop_duplicates(inplace=True)
tags_clean["timestamp"] = pd.to_datetime(tags_clean["timestamp"], unit="s")
tags_clean.rename(columns={"timestamp": "tagged_at"}, inplace=True)
tags_clean["tag"] = tags_clean["tag"].str.strip().str.lower()
tags_clean.dropna(subset=["tag"], inplace=True)

print(f"tags rows: {len(tags):,}  →  {len(tags_clean):,} after cleaning")


## 6. Feature Engineering

In [ ]:
# ── Movie-level aggregates ──
movie_stats = (
    ratings_clean
    .groupby("movieId")
    .agg(
        avg_rating    = ("rating", "mean"),
        rating_count  = ("rating", "count"),
        rating_std    = ("rating", "std"),
        first_rated   = ("rated_at", "min"),
        last_rated    = ("rated_at", "max"),
    )
    .reset_index()
)
movie_stats["avg_rating"]   = movie_stats["avg_rating"].round(4)
movie_stats["rating_std"]   = movie_stats["rating_std"].fillna(0).round(4)

# Bayesian popularity score (Bayes average — guards against small-N movies)
C = movie_stats["rating_count"].mean()       # mean number of ratings
m = movie_stats["avg_rating"].mean()         # global mean rating
movie_stats["popularity_score"] = (
    (movie_stats["rating_count"] / (movie_stats["rating_count"] + C)) * movie_stats["avg_rating"]
    + (C / (movie_stats["rating_count"] + C)) * m
).round(4)

print("Movie stats sample:")
display(movie_stats.sort_values("popularity_score", ascending=False).head(10))


In [ ]:
# ── Genre encoding (multi-hot) ──
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
genre_matrix = pd.DataFrame(
    mlb.fit_transform(movies_clean["genre_list"]),
    columns=mlb.classes_,
    index=movies_clean.index
)
genre_matrix.insert(0, "movieId", movies_clean["movieId"].values)

print(f"Genre columns ({len(mlb.classes_)}): {list(mlb.classes_)}")
display(genre_matrix.head(3))


In [ ]:
# ── Recency feature ──
latest_date = ratings_clean["rated_at"].max()
ratings_clean["days_since_rated"] = (latest_date - ratings_clean["rated_at"]).dt.days

# User-level stats
user_stats = (
    ratings_clean
    .groupby("userId")
    .agg(
        user_rating_count  = ("rating", "count"),
        user_avg_rating    = ("rating", "mean"),
        user_rating_std    = ("rating", "std"),
    )
    .reset_index()
)
user_stats["user_avg_rating"] = user_stats["user_avg_rating"].round(4)
user_stats["user_rating_std"] = user_stats["user_rating_std"].fillna(0).round(4)

print("User stats sample:")
display(user_stats.head(5))


## 7. User-Item Interaction Matrix

In [ ]:
# Dense pivot (for small dataset — fine at 100K scale)
user_item_matrix = ratings_clean.pivot_table(
    index="userId", columns="movieId", values="rating"
)

n_users, n_movies = user_item_matrix.shape
n_ratings = ratings_clean.shape[0]
sparsity = 1 - (n_ratings / (n_users * n_movies))

print(f"Matrix shape  : {n_users} users × {n_movies} movies")
print(f"Total ratings : {n_ratings:,}")
print(f"Sparsity      : {sparsity:.4%}")
print()
print("This is the 'cold start' challenge: new users/movies have no history.")


In [ ]:
# Sparse version (CSR) — what the modeling team will likely use
user_item_sparse = csr_matrix(user_item_matrix.fillna(0).values)
print(f"CSR matrix: {user_item_sparse.shape}, {user_item_sparse.nnz:,} stored values")


## 8. Train / Validation / Test Split

> **Why this matters for recommenders:**  
> We split *by interaction*, not by user or movie. Each rating is a data point. We need the model to generalize to *unseen interactions*, not unseen users. A random split simulates that.
>
> Ratio: **80% train / 10% validation / 10% test**


In [ ]:
train_val, test = train_test_split(
    ratings_clean, test_size=0.10, random_state=42, shuffle=True
)
train, val = train_test_split(
    train_val, test_size=0.111, random_state=42   # 0.111 * 0.9 ≈ 0.10
)

print(f"Train : {len(train):,} rows  ({len(train)/len(ratings_clean):.1%})")
print(f"Val   : {len(val):,}  rows  ({len(val)/len(ratings_clean):.1%})")
print(f"Test  : {len(test):,}  rows  ({len(test)/len(ratings_clean):.1%})")


## 9. Save Clean Artifacts

In [ ]:
PROCESSED = "../data/processed"
os.makedirs(PROCESSED, exist_ok=True)

# Core interaction files
train.to_csv(f"{PROCESSED}/train.csv", index=False)
val.to_csv(f"{PROCESSED}/val.csv", index=False)
test.to_csv(f"{PROCESSED}/test.csv", index=False)
ratings_clean.to_csv(f"{PROCESSED}/ratings_clean.csv", index=False)

# Movie metadata + features
movies_clean.to_csv(f"{PROCESSED}/movies_clean.csv", index=False)
movie_stats.to_csv(f"{PROCESSED}/movie_stats.csv", index=False)
genre_matrix.to_csv(f"{PROCESSED}/genre_encoded.csv", index=False)

# User features
user_stats.to_csv(f"{PROCESSED}/user_stats.csv", index=False)

# Full user-item matrix (dense)
user_item_matrix.to_csv(f"{PROCESSED}/user_item_matrix.csv")

print("Saved to /data/processed:")
for f in sorted(os.listdir(PROCESSED)):
    size = os.path.getsize(f"{PROCESSED}/{f}") / 1024
    print(f"  {f:<35}  {size:6.1f} KB")


## 10. Summary for Team

In [ ]:
summary = {
    "Total ratings (clean)"  : len(ratings_clean),
    "Unique users"           : ratings_clean["userId"].nunique(),
    "Unique movies"          : ratings_clean["movieId"].nunique(),
    "Rating range"           : f"{ratings_clean['rating'].min()} – {ratings_clean['rating'].max()}",
    "Matrix sparsity"        : f"{sparsity:.4%}",
    "Train rows"             : len(train),
    "Val rows"               : len(val),
    "Test rows"              : len(test),
    "Genres encoded"         : len(mlb.classes_),
}

print("=" * 45)
print("  CLEAN DATA SUMMARY")
print("=" * 45)
for k, v in summary.items():
    print(f"  {k:<28}: {v}")
print("=" * 45)
print("\nAll artifacts saved to /data/processed/ ✓")
